<!-- # Intro To Agents Part 1: Tools, MCPs & Tracing
--------------------

1. Introduction

2. Agents & Tools with LangChain

3. Model Context Protocols (MCPs) with FastMCP

4. Agents & Tracing with LangSmith

5. Next Steps -->

In [ ]:
# import sys
# import os
# from dotenv import load_dotenv
# load_dotenv("..")


False

<!--  -->

<!-- ## 1. Introduction  -->

In [ ]:
# from langchain_groq import ChatGroq
# from langchain_core.output_parsers import StrOutputParser
# from langchain.agents import create_agent

# model = ChatGroq(model="openai/gpt-oss-120b")
# llm  = model | StrOutputParser() 

# llm.invoke("Hello")

/Users/mikeharmon/Desktop/mcpweather/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


'Hello! How can I assist you today?'

In [ ]:
# print(llm.invoke("What is the weather like in New York City today?"))

I’m not able to pull live weather data, so I can’t give you the exact conditions for New York City right now.  

**Quick ways to get the current forecast:**

| Option | How to use it |
|--------|----------------|
| **Weather websites** | Visit sites like [weather.com](https://weather.com), [AccuWeather](https://www.accuweather.com), or the National Weather Service’s page for New York, NY. |
| **Smartphone apps** | Open the default weather app on iOS/Android, or download apps such as The Weather Channel, Weather Underground, or CARROT Weather. |
| **Voice assistants** | Ask Siri, Google Assistant, or Alexa “What’s the weather in New York City?” |
| **Search engines** | Type “New York City weather” into Google, Bing, or DuckDuckGo – the search results page shows a live summary. |
| **Local news** | Check the weather section of NY‑based news outlets (e.g., NY Times, NBC New York) or their TV broadcasts. |

---

### Typical late‑August weather in NYC
If you’re looking for a general idea of

<!-- ## 2. Agents & Tools with LangChain -->

In [ ]:
# sys.path.append('..')
# from mcp.server import get_weather

ImportError: cannot import name 'get_weather' from 'mcp.server' (/Users/mikeharmon/Desktop/mcpweather/.venv/lib/python3.13/site-packages/mcp/server/__init__.py)

In [ ]:
# results = get_weather("New York")

In [ ]:
# results

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 802,
   'main': 'Clouds',
   'description': 'scattered clouds',
   'icon': '03d'}],
 'base': 'stations',
 'main': {'temp': 27.95,
  'feels_like': 28.76,
  'temp_min': 26.64,
  'temp_max': 29.06,
  'pressure': 1009,
  'humidity': 54,
  'sea_level': 1009,
  'grnd_level': 1007},
 'visibility': 10000,
 'wind': {'speed': 3.09, 'deg': 240},
 'clouds': {'all': 29},
 'dt': 1786579056,
 'sys': {'type': 1,
  'id': 4610,
  'country': 'US',
  'sunrise': 1786529005,
  'sunset': 1786579133},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

In [ ]:
# from langchain.tools import tool


In [ ]:
# weather_tool = tool("get_weather", get_weather)

In [ ]:
# type(weather_tool)

langchain_core.tools.structured.StructuredTool

In [ ]:
# await weather_tool.ainvoke("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 802,
   'main': 'Clouds',
   'description': 'scattered clouds',
   'icon': '03d'}],
 'base': 'stations',
 'main': {'temp': 27.95,
  'feels_like': 28.76,
  'temp_min': 26.64,
  'temp_max': 29.06,
  'pressure': 1009,
  'humidity': 54,
  'sea_level': 1009,
  'grnd_level': 1007},
 'visibility': 10000,
 'wind': {'speed': 3.09, 'deg': 240},
 'clouds': {'all': 29},
 'dt': 1786579056,
 'sys': {'type': 1,
  'id': 4610,
  'country': 'US',
  'sunrise': 1786529005,
  'sunset': 1786579133},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

In [ ]:
agents = create_agent(model=model, tools=[weather_tool])

NameError: name 'weather_tool' is not defined

In [ ]:
query = "What is the weather in New York?"

In [ ]:
result = await agents.ainvoke({'messages': [{'role': 'user', 'content': query}]})

In [ ]:
messages = result.get("messages")

In [ ]:
messages

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='e4f328cb-49f0-4116-80e5-377bd24bcb5f'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in New York. We need to call the function get_weather with city "New York". Use default API key.', 'tool_calls': [{'id': 'fc_bcbcbb64-0db0-438f-a5ea-d3edee0cfb66', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 261, 'total_tokens': 318, 'completion_time': 0.120231658, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.147129581, 'prompt_tokens_details': None, 'queue_time': 0.212004861, 'total_time': 0.267361239}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ff86

<!-- ## 3. Model Context Protocols (MCPs) with FastMCP -->

In [6]:
from fastmcp import Client

In [36]:
mcp_client = Client("http://localhost:8000/mcp")

In [37]:
async with mcp_client:
    tools = await mcp_client.list_tools()

In [38]:
tools

[Tool(name='get_weather', title=None, description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', inputSchema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, outputSchema={'additionalProperties': True, 'type': 'object'}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None),
 Tool(name='convert_address_to_point', title=None, description='Convert a street address to a Shapely Point object.', inputSchema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'description': 'A free‑form US address (e

In [39]:
async with mcp_client:
    result = await mcp_client.call_tool("get_weather", {"city": "New York"})

In [40]:
async with mcp_client:
    point = await mcp_client.call_tool("convert_address_to_point", {"address": "567 Ocean Avenue, Brooklyn, NY 11226"})

In [41]:
point

CallToolResult(content=[TextContent(type='text', text='{"lat":40.6488957,"lng":-73.9603205}', annotations=None, meta=None)], structured_content={'lat': 40.6488957, 'lng': -73.9603205}, meta=None, data={'lat': 40.6488957, 'lng': -73.9603205}, is_error=False)

AttributeError: 'Client' object has no attribute 'get_tool'

In [ ]:
mcp_client.tool

AttributeError: 'Client' object has no attribute 'get_tool_schema'

In [ ]:
# json.dumps(result.content[0].text)

'{"lat":40.6488957,"lng":-73.9603205}'

In [ ]:
# result

CallToolResult(content=[TextContent(type='text', text='{"lat":40.6488957,"lng":-73.9603205}', annotations=None, meta=None)], structured_content={'lat': 40.6488957, 'lng': -73.9603205}, meta=None, data={'lat': 40.6488957, 'lng': -73.9603205}, is_error=False)

In [27]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": "http://localhost:8000/mcp", "transport": "http"}})

In [28]:
# 

In [29]:
tool_list = await lc_client.get_tools()

In [30]:
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10eb3cc20>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'descrip

In [31]:
weather_mcp_tool = [tool_list[0]]

In [32]:
weather_mcp_tool

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10eb3cc20>)]

In [33]:
weather_agent = create_agent(model=model, tools=weather_mcp_tool)

In [34]:
result = await weather_agent.ainvoke({'messages': [{
    'role': 'user', 'content': 'What is the weather in New York?'}
    ]})

In [35]:
result.get("messages")

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='5feaec51-cbdd-4ec4-8f79-531bc24c7b4c'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in New York. Use function get_weather.', 'tool_calls': [{'id': 'fc_34ad84e1-ac1e-4b98-a4a6-33ece87b3040', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 196, 'total_tokens': 238, 'completion_time': 0.087751453, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007896551, 'prompt_tokens_details': None, 'queue_time': 0.173119734, 'total_time': 0.095648004}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5781dfb07c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a01284-4415-7881-944d-9d17248b932c-0', tool_calls=[{'name': 'get_we

<!-- ## 4. Agents & Traces With LangSmith


Others tools in MCP Server -->

In [ ]:
# tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10eb8cae0>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'descrip

In [25]:
agent = create_agent(model=model, tools=tool_list)

In [26]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattan, manhattan in?'
}]})

RuntimeError: Invalid structured content returned by tool get_police_precinct: None is not of type 'integer'

Failed validating 'type' in schema['properties']['result']:
    {'type': 'integer'}

On instance['result']:
    None

In [ ]:
# result.get("messages")

[HumanMessage(content='What police precinct is 306 West 54th Street, manhattan, manhattan in?', additional_kwargs={}, response_metadata={}, id='2fda0f57-58d7-4aa3-a12c-7dbfb90e3f9f'),
 AIMessage(content='', additional_kwargs={'reasoning_content': "We need to find police precinct for address 306 West 54th Street, Manhattan. Use convert_address_to_point then get_police_precinct. Let's call convert_address_to_point.", 'tool_calls': [{'id': 'fc_c475a560-2b91-44c9-88e6-1aeabc7b27aa', 'function': {'arguments': '{"address":"306 West 54th Street, Manhattan, NY"}', 'name': 'convert_address_to_point'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 427, 'total_tokens': 503, 'completion_time': 0.163319927, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.020054843, 'prompt_tokens_details': None, 'queue_time': 0.204392424, 'total_time': 0.18337477}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a19b1544c'

In [ ]:
# print(result.get("messages")[-1].content)

The address **306 West 54th Street, Manhattan, NY** falls within **NYPD Precinct 18 (Midtown North Precinct)**.  

- **Borough:** Manhattan  
- **Precinct:** Midtown North Precinct  
- **Phone:** 212‑767‑8400  
- **Address:** 306 West 54th Street, New York, NY 10019  

If you need any additional details—such as hours of operation, nearby police stations, or how to contact the precinct—just let me know!


In [ ]:
# result = await agent.ainvoke({
#     'messages': [{'role': 'user', 'content': 'Where is the closet bathroom to 625 Atlantic Ave, Brooklyn'
# }]})

ToolException: Output validation error: outputSchema defined but no structured output returned

In [ ]:
# print(result.get("messages")[-1].content)

AttributeError: 'CallToolResult' object has no attribute 'get'

<!--  -->

In [ ]:
# 